In [57]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

In [58]:
date = "current"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

In [59]:
end_use = "naics?get=CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url) 
    
print(r)
    
df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

df.columns = r.json()[0]

df["total_imports"] = df["CON_VAL_MO"].astype(float)

df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])

top_products = grp.agg({"total_imports":"sum","CTY_CODE":"first"})

country_list = list(top_products.sort_values(by = "total_imports", ascending = False).CTY_CODE)[0:31]


['TOTAL FOR ALL COUNTRIES','NAFTA','EUROPEAN UNION']

<Response [200]>


['TOTAL FOR ALL COUNTRIES', 'NAFTA', 'EUROPEAN UNION']

In [60]:
df.tail()

,CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL,time,total_imports
38796,3656796,7940,ZAMBIA,DET,2025-10,3656796.0
38797,1146041,7950,ESWATINI,DET,2025-10,1146041.0
38798,2561320,7960,ZIMBABWE,DET,2025-10,2561320.0
38799,2339274,7970,MALAWI,DET,2025-10,2339274.0
38801,8071774,7990,LESOTHO,DET,2025-10,8071774.0


In [61]:
country_list[0] = ""

In [62]:
country_list.extend(["0003", "0020"])

In [63]:
len(country_list)

33

In [65]:
end_use = "hs?get=CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&time=" + "from+2013-01" + "&COMM_LVL=HS10" 

for xxx in country_list:
    
    out_file = ".\\data"+ "\\imports-hs10\\" + xxx + "data-" + date + ".parquet"
    
    if xxx == "":
        out_file = ".\\data"+ "\\imports-hs10\\" + "TOTAL" + "data-" + date + ".parquet"
    
    
    if os.path.exists(out_file):
        
        print("Already have downloaded file")
        
        continue
    
    print(xxx)
    
    url = surl + "&CTY_CODE=" + xxx
    
    if xxx == "":
        url = surl
    
    r = requests.get(url) 
    
    print(r)
    
    foo = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

    foo.columns = r.json()[0]

    pq.write_table(pa.Table.from_pandas(foo), out_file)

Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
Already have downloaded file
0003
<Response [200]>
0020
<Response [200]>


In [66]:
foo.head()

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,time,COMM_LVL,CTY_CODE
0,USMCA (NAFTA),6799377,0,8805290000,"GROUND FLYING TRAINERS AND PARTS THEREOF, NESOI",2013-01,HS10,0020
1,USMCA (NAFTA),1682901,0,8903990500,CANOES (NORMALLY NOT USED WITH MOTORS OR SAILS),2013-01,HS10,0020
2,USMCA (NAFTA),280617,0,8903991500,ROW BOATS(NORMALLY NOT USED WITH MOTRS/SAILS),2013-01,HS10,0020
3,USMCA (NAFTA),2150,43,9002204000,FILTERS AND PARTS FOR PHOTOGRAPHIC USE,2013-01,HS10,0020
4,USMCA (NAFTA),60732,762,9002208000,FILTERS AND PARTS EXCEPT PHOTOGRAPHIC,2013-01,HS10,0020


In [67]:
foo.tail()

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,time,COMM_LVL,CTY_CODE
1836155,USMCA (NAFTA),1793055,0,3907500000,ALKYD RESINS,2025-10,HS10,0020
1836156,USMCA (NAFTA),18225400,0,3907610010,POLYETHYLENE TEREPHTHALATE VISC GT=78 ML/G LT=88,2025-10,HS10,0020
1836157,USMCA (NAFTA),4799526,0,3907610050,POLYETHYLENE TEREPHTHALATE VISCSTY GT 88 ML/G,2025-10,HS10,0020
1836158,USMCA (NAFTA),2721085,0,3907690010,"OTHR POLYETHYLENE TEREPHTHALATE VIS GT=70ML/G,...",2025-10,HS10,0020
1836159,USMCA (NAFTA),4506380,4973,3907690050,OTHER POLYETHYLENE TEREPHTHALATE VISCOS GT= 78...,2025-10,HS10,0020
